# ขั้นที่ 2: Forecast — พยากรณ์ยอดขาย 28 วันข้างหน้า

## ภาพรวมของขั้นนี้

| | |
|---|---|
| **คำถาม** | อีก 28 วันข้างหน้า สินค้าแต่ละรายการในแต่ละสาขาจะขายได้กี่ชิ้นต่อวัน |
| **ทำไมต้องรู้** | ใช้ตัดสินใจว่าจะสั่งของเท่าไร ถ้าทายต่ำไปของขาด ถ้าทายสูงไปของล้น (ผลกระทบเป็นเงินดูในขั้นที่ 3) |
| **ขนาดงาน** | 30,490 รายการ (สินค้า × สาขา) × 28 วัน = พยากรณ์ 853,720 ค่า |

## ขั้นตอนการทำงาน

```
2.1 วิธีวัดผล         ใช้ WRMSSE + ทดสอบย้อนหลัง 3 รอบ (backtest)
      ↓
2.2 Feature engineering   แปลงข้อมูลดิบเป็นตัวแปรที่โมเดลใช้ได้ (จาก insight ขั้นที่ 1)
      ↓
2.3 Baseline           วิธีง่ายๆ ที่โมเดลต้องเอาชนะให้ได้
      ↓
2.4 โมเดล ML            LightGBM, XGBoost, CatBoost, Random Forest
      ↓
2.5 Backtest           เทรนและวัดผลทุกวิธีบนข้อมูลชุดเดียวกัน
      ↓
2.6 ผลลัพธ์             ภาพรวม → แยกหมวด → แยกระดับ → รายสินค้า
      ↓
2.7 อธิบายโมเดล         ปัจจัยไหนสำคัญที่สุด (SHAP)
      ↓
2.8 พยากรณ์จริง         28 วันข้างหน้า (23 พ.ค. – 19 มิ.ย. 2016)
```

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True

FIGURES = ROOT / 'reports' / 'figures'
ANALYSIS = ROOT / 'analysis'
sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', '{:,.2f}'.format)


def show(fig, name):
    """แสดงกราฟในหน้านี้ และบันทึกสำเนาไว้ใน reports/figures"""
    fig.tight_layout()
    fig.savefig(FIGURES / f'{name}.png', dpi=150, bbox_inches='tight')
    plt.show()
import time

import lightgbm as lgb
import matplotlib.dates as mdates
import shap
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor

from src.data import PROCESSED, ID_COLS, load_long, load_raw
from src.features import CAT_FEATURES, HORIZON, FeatureBuilder
from src.metrics import WRMSSE

In [ ]:
FOLDS = [1857, 1885, 1913]  # วันสุดท้ายของข้อมูลที่ใช้เทรนในแต่ละรอบ backtest
FINAL_END = 1941             # ข้อมูลจริงวันสุดท้าย ใช้เทรนตัวจริงเพื่อพยากรณ์ d_1942–d_1969
TRAIN_DAYS = 365             # เทรนด้วยประวัติ 1 ปีล่าสุด (ลดเวลาและ RAM)
N_ROUNDS = 800               # จำนวนต้นไม้ของโมเดล boosting
rng = np.random.default_rng(42)

sales, calendar, prices = load_raw()
fb = FeatureBuilder(load_long(), calendar)
Y = fb.sales  # ตารางยอดขาย (สินค้า×สาขา, วัน) คอลัมน์ j คือวัน d_(j+1)
d_cols = [c for c in sales.columns if c.startswith('d_')]
dates = pd.to_datetime(calendar['date']).to_numpy()
print(f'ยอดขาย {Y.shape[0]:,} รายการ × {Y.shape[1]:,} วัน')

---
## 2.1 วิธีวัดผล

### ตัวชี้วัด: WRMSSE (ยิ่งต่ำยิ่งดี)

อธิบายแบบง่ายใน 3 ข้อ
1. **เทียบกับวิธีง่ายที่สุด:** วัด error ของการพยากรณ์ หารด้วย error ของวิธี "ทายว่าวันนี้ขายเท่าเมื่อวาน" ถ้าได้ค่า **ต่ำกว่า 1 แปลว่าดีกว่าวิธีง่ายๆ**
2. **ให้น้ำหนักตามรายได้:** สินค้าที่ทำเงินมากมีผลต่อคะแนนมาก เพราะถ้าทายผิดจะเสียหายมาก
3. **วัดทุกระดับ:** วัดทั้งยอดรวมบริษัท รายรัฐ รายสาขา รายหมวด ... ลงไปถึงรายสินค้า×สาขา รวม 12 ระดับ แล้วเฉลี่ย โมเดลจึงต้องแม่นทั้งภาพรวมและรายตัว

### ทดสอบย้อนหลัง (backtest)

แกล้งทำเป็นว่าอยู่ในอดีต เทรนโมเดลด้วยข้อมูลถึงวันหนึ่ง แล้วทาย 28 วันถัดไปซึ่งรู้คำตอบจริงอยู่แล้ว
ทำ 3 รอบในช่วงเวลาต่างกันเพื่อให้ผลน่าเชื่อถือ **ห้ามสุ่มแบ่งข้อมูล** เพราะจะกลายเป็นใช้อนาคตทายอดีต

In [ ]:
fig, ax = plt.subplots(figsize=(13, 2.8))
rows = [(f'Backtest {i + 1}', end, 'darkorange') for i, end in enumerate(FOLDS)] + [('Final forecast', FINAL_END, 'seagreen')]
day_num = mdates.date2num(dates)  # วันที่ในหน่วย "วัน" ของ matplotlib ใช้กำหนดความกว้างแท่งได้
for i, (label, end, color) in enumerate(rows):
    ax.barh(i, TRAIN_DAYS, left=day_num[end - TRAIN_DAYS], color='steelblue')
    ax.barh(i, HORIZON, left=day_num[end], color=color)
    ax.text(day_num[end + HORIZON - 1] + 4, i, f'{"forecast" if end == FINAL_END else "test"} {pd.Timestamp(dates[end]):%d %b} – {pd.Timestamp(dates[end + HORIZON - 1]):%d %b %Y}', va='center')
ax.set_yticks(range(len(rows)), [r[0] for r in rows])
ax.invert_yaxis()
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in ['steelblue', 'darkorange', 'seagreen']],
          labels=['Train (365 days)', 'Test (28 days, actual known)', 'Forecast (future, unknown)'], loc='lower left', fontsize=8)
ax.xaxis_date()
ax.set(title='Backtest design', xlim=(day_num[FOLDS[0] - TRAIN_DAYS - 10], day_num[-1] + 75))
show(fig, '02_1_backtest_design')

---
## 2.2 Feature engineering

โมเดลไม่สามารถอ่าน "ตารางยอดขาย" ได้โดยตรง ต้องแปลงเป็นตัวแปร (feature) ที่บอกข้อมูลแต่ละวันของแต่ละสินค้า
feature ทุกตัวมาจาก insight ในขั้นที่ 1

| กลุ่ม | Feature | ความหมาย | มาจาก insight |
|---|---|---|---|
| **ยอดขายในอดีต (lag)** | `lag_28`, `lag_35`, ..., `lag_56` | ยอดขายเมื่อ 28, 35, ..., 56 วันก่อน | ยอดขายมีรูปแบบซ้ำรายสัปดาห์ (1.1) |
| **ค่าเฉลี่ยเคลื่อนที่ (rolling)** | `rmean_7`, `rmean_28`, `rmean_56` | ยอดขายเฉลี่ย 7/28/56 วัน นับถอยหลังจาก 28 วันก่อน | บอกระดับยอดขายปกติและแนวโน้ม (1.1, 1.5) |
| **ราคา** | `sell_price`, `price_rel_regular`, `price_change_7d` | ราคาปัจจุบัน, ราคาเทียบราคาปกติ, ราคาเปลี่ยนจากสัปดาห์ก่อนกี่ % | ลดราคาแล้วขายดีขึ้น (1.4.4) |
| **ปฏิทิน** | `wday`, `month`, `mday` | วันในสัปดาห์, เดือน, วันที่ของเดือน | เสาร์-อาทิตย์ขายดี (1.4.1) |
| **Event / SNAP** | `event_name_1`, `event_type_1`, `snap` | วันนั้นมีเทศกาลอะไร, เป็นวัน SNAP ของรัฐนั้นไหม | ยอดเปลี่ยนในวัน event และวัน SNAP (1.4.2–1.4.3) |
| **รหัสสินค้า/สถานที่** | `item_id`, `dept_id`, `cat_id`, `store_id`, `state_id` | สินค้าอะไร อยู่สาขาไหน | แต่ละสาขาและหมวดขายต่างกัน (1.2–1.3) |

### ทำไม lag ต้องเริ่มที่ 28 วัน

ต้องการทาย 28 วันข้างหน้า **ในครั้งเดียว** ถ้าใช้ยอดขาย "เมื่อวาน" เป็น feature จะทายได้แค่วันแรก เพราะวันที่ 2–28 ยังไม่รู้ยอดของเมื่อวาน
การใช้ข้อมูลที่เก่ากว่า 28 วันทำให้ทุกวันในช่วงพยากรณ์มี feature ครบ เรียกวิธีนี้ว่า **direct forecasting** ข้อดีคือ error ไม่สะสม

ตัวอย่างตาราง feature ที่โมเดลเห็น (1 แถว = สินค้า 1 รายการ ใน 1 สาขา ใน 1 วัน)

In [ ]:
d_from = FINAL_END - 179
window = slice(d_from - 1, FINAL_END)
with np.errstate(invalid='ignore'):
    price_changed = np.nanmax(fb.price[:, window], axis=1) > np.nanmin(fb.price[:, window], axis=1) * 1.02
revenue = (Y[:, window] * np.nan_to_num(fb.price[:, window])).sum(axis=1)
top_idx = int(np.argmax(np.where(price_changed, revenue, -1)))  # รายได้สูงสุดในกลุ่มที่มีการเปลี่ยนราคา
top_name = f"{sales.loc[top_idx, 'item_id']} @ {sales.loc[top_idx, 'store_id']}"
X_demo, y_demo = fb.build(d_from, FINAL_END)
one = X_demo.iloc[top_idx * 180:(top_idx + 1) * 180].assign(sales=y_demo[top_idx * 180:(top_idx + 1) * 180])
one.index = pd.DatetimeIndex(dates[d_from - 1:FINAL_END], name='date')
print(f'ตัวอย่าง: {top_name} (รายได้สูงสุดในกลุ่มสินค้าที่มีการเปลี่ยนราคาใน 180 วัน) | feature ทั้งหมด {X_demo.shape[1]} ตัว')
one.tail(5).T

ตัวอย่างการทำงานของ feature กับสินค้าตัวเดียวกัน (180 วันล่าสุด)
- **กราฟบน:** ยอดขายจริง (แท่ง) เทียบกับ `lag_28` และ `rmean_28` จะเห็นว่า rolling mean จับระดับยอดขายได้ดี ส่วน lag เก็บรูปแบบรายสัปดาห์ไว้
- **กราฟล่าง:** ราคาเทียบกับราคาปกติ (1.0 = ราคาปกติ, ต่ำกว่า 1 = ลดราคา) ให้สังเกตยอดขายในกราฟบนช่วงที่ราคาเปลี่ยน

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
axes[0].bar(one.index, one['sales'], color='lightgray', label='Actual sales')
axes[0].plot(one.index, one['lag_28'], color='darkorange', lw=1, label='lag_28 (sales 28 days ago)')
axes[0].plot(one.index, one['rmean_28'], color='steelblue', lw=2, label='rmean_28 (28-day avg, ending 28 days ago)')
axes[0].set(title=f'Sales features: {top_name}', ylabel='Units')
axes[0].legend(loc='upper left', fontsize=8)
axes[1].plot(one.index, one['price_rel_regular'], color='seagreen', drawstyle='steps-post')
axes[1].set(title='price_rel_regular (1.0 = regular price)', ylabel='Ratio')
show(fig, '02_2_feature_example')

**สิ่งที่เห็นจากตัวอย่างนี้**
- `rmean_28` ตามระดับยอดขายได้ดี แต่ช้ากว่ายอดจริง 28 วันตามที่ออกแบบไว้
- ช่วงต้นเดือนมีนาคม ยอดขายเป็น 0 ติดกันเกือบ 3 สัปดาห์ ทั้งที่ปกติขายได้วันละ 50–100 ชิ้น และตรงกับช่วงที่ลดราคา จึงน่าจะเกิดจาก **ของหมดสต็อก** ไม่ใช่ไม่มีคนซื้อ (censored demand)
- 28 วันต่อมา `lag_28` และ `rmean_28` ตกลงตาม โมเดลจึงอาจทายต่ำเกินไปในช่วงนั้น นี่คือข้อจำกัดของข้อมูลที่ไม่มีระดับสต็อก (กล่าวถึงอีกครั้งในขั้นที่ 4)

**feature ไหนสัมพันธ์กับยอดขายมากที่สุด**

ดูความสัมพันธ์แบบ Spearman ระหว่าง feature ตัวเลขแต่ละตัวกับยอดขายจริง (สุ่ม 300,000 แถว)
เป็นการตรวจสอบเบื้องต้นก่อนเทรนโมเดล ส่วนความสำคัญจริงของ feature ในโมเดลดูได้ที่หัวข้อ 2.7

In [ ]:
numeric = [c for c in X_demo.columns if c not in CAT_FEATURES]
sample_rows = rng.choice(len(X_demo), 300_000, replace=False)
corr = X_demo.iloc[sample_rows][numeric].corrwith(pd.Series(y_demo[sample_rows]), method='spearman').sort_values()

fig, ax = plt.subplots(figsize=(9, 5))
corr.plot.barh(ax=ax, color=np.where(corr > 0, 'steelblue', 'indianred'))
ax.set(title='Spearman correlation of each feature with daily sales', xlabel='Correlation')
show(fig, '02_2_feature_correlation')
del X_demo, y_demo

**ข้อสังเกต:** feature ยอดขายในอดีต (rolling mean และ lag) สัมพันธ์กับยอดขายสูงที่สุด เพราะสินค้าที่เคยขายดีมักจะขายดีต่อ
ส่วนปฏิทิน ราคา และ SNAP สัมพันธ์น้อยกว่าเมื่อดูทีละตัว แต่ช่วยอธิบายว่า **วันไหน** จะขายมากหรือน้อยกว่าระดับปกติ

---
## 2.3 Baseline: วิธีง่ายๆ ที่โมเดลต้องเอาชนะ

| วิธี | ทายอย่างไร |
|---|---|
| Naive | ทุกวันขายเท่ากับวันสุดท้ายที่รู้ |
| Seasonal naive | ทำซ้ำยอดขาย 7 วันสุดท้าย (จับรูปแบบรายสัปดาห์) |
| Mean 28d | ค่าเฉลี่ย 28 วันล่าสุด |

ถ้าโมเดล ML ชนะ baseline ไม่ได้ ก็ไม่คุ้มที่จะใช้

In [ ]:
def naive(end):
    return np.repeat(Y[:, end - 1:end], HORIZON, axis=1)


def seasonal_naive(end):
    return np.tile(Y[:, end - 7:end], HORIZON // 7)


def mean_28d(end):
    return np.repeat(Y[:, end - 28:end].mean(axis=1, keepdims=True), HORIZON, axis=1)


BASELINES = {'Naive': naive, 'Seasonal naive': seasonal_naive, 'Mean 28d': mean_28d}

---
## 2.4 โมเดล Machine Learning

ใช้โมเดลแบบต้นไม้ตัดสินใจ (tree-based) 4 แบบ ทุกโมเดลใช้ feature ชุดเดียวกัน และเป็น **โมเดลเดียวที่เรียนจากทุกสินค้าพร้อมกัน** (global model)
เพราะสินค้าส่วนใหญ่ขายช้า ถ้าแยกโมเดลรายสินค้าจะมีข้อมูลน้อยเกินไป

| โมเดล | หลักการ | เหตุผลที่เลือก |
|---|---|---|
| **LightGBM** | Gradient boosting: สร้างต้นไม้ทีละต้น แต่ละต้นแก้ error ของต้นก่อนหน้า | เร็วที่สุดกับข้อมูลหลายล้านแถว |
| **XGBoost** | Gradient boosting เหมือนกัน แต่วิธีสร้างต้นไม้ต่างกัน | แม่นและเสถียร ใช้เปรียบเทียบกับ LightGBM |
| **CatBoost** | Gradient boosting ที่จัดการตัวแปรประเภทหมวดหมู่ได้เอง | ไม่ต้องแปลงรหัสสินค้าและสาขาเอง |
| **Random Forest** | สร้างต้นไม้หลายต้นอิสระกันแล้วเฉลี่ยผล | เข้าใจง่าย ใช้เป็นตัวเทียบฝั่ง bagging |

โมเดล boosting ใช้ **Tweedie loss** ซึ่งเหมาะกับยอดขายที่เป็น 0 บ่อย (จากหัวข้อ 1.5)
CatBoost และ Random Forest ช้ากว่ามาก จึงเทรนบนข้อมูลที่สุ่มมาบางส่วน

In [ ]:
def ml_forecast(make_model, end, fit_kwargs=None, sample=None):
    """เทรนด้วยข้อมูลถึงวัน end แล้วทาย end+1..end+28 คืนค่า (pred, model, X_test)"""
    X, y = fb.build(end - TRAIN_DAYS + 1, end)
    on_sale = X['sell_price'].notna().to_numpy()  # เทรนเฉพาะวันที่สินค้าวางขายแล้ว
    X, y = X[on_sale], y[on_sale]
    if sample and sample < len(y):
        idx = rng.choice(len(y), sample, replace=False)
        X, y = X.iloc[idx], y[idx]

    model = make_model()
    model.fit(X, y, **(fit_kwargs or {}))

    X_test, _ = fb.build(end + 1, end + HORIZON)
    pred = model.predict(X_test).reshape(-1, HORIZON)
    pred[X_test['sell_price'].isna().to_numpy().reshape(-1, HORIZON)] = 0  # สินค้ายังไม่วางขาย
    return np.clip(pred, 0, None), model, X_test


MODELS = {
    'LightGBM': lambda end: ml_forecast(
        lambda: lgb.LGBMRegressor(
            objective='tweedie', tweedie_variance_power=1.1, n_estimators=N_ROUNDS, learning_rate=0.05,
            num_leaves=127, min_child_samples=100, subsample=0.5, subsample_freq=1, colsample_bytree=0.8, verbose=-1,
        ),
        end, fit_kwargs={'categorical_feature': CAT_FEATURES},
    ),
    'XGBoost': lambda end: ml_forecast(
        lambda: xgb.XGBRegressor(
            objective='reg:tweedie', tweedie_variance_power=1.1, tree_method='hist', n_estimators=N_ROUNDS,
            learning_rate=0.05, max_depth=8, subsample=0.5, colsample_bytree=0.8, max_bin=128,
        ),
        end,
    ),
    'CatBoost': lambda end: ml_forecast(
        lambda: CatBoostRegressor(
            loss_function='Tweedie:variance_power=1.1', iterations=N_ROUNDS, learning_rate=0.1, depth=8,
            verbose=0, allow_writing_files=False,
        ),
        end, fit_kwargs={'cat_features': CAT_FEATURES}, sample=3_000_000,
    ),
    'Random Forest': lambda end: ml_forecast(
        lambda: RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=50, max_features=0.5, n_jobs=-1),
        end, sample=1_000_000,
    ),
}

---
## 2.5 Backtest: เทรนและวัดผลทุกวิธี

แต่ละรอบวัด WRMSSE ทั้งภาพรวมและรายหมวด พร้อม **bias** (ผลรวมที่ทายสูงหรือต่ำกว่ายอดจริงกี่ %)
ถ้า bias ติดลบแปลว่าทายต่ำเกินไปอย่างเป็นระบบ ซึ่งจะทำให้ของขาด

In [ ]:
rows, last_fold = [], {}
for end in FOLDS:
    train_wide = sales[ID_COLS + d_cols[:end]]
    y_true = Y[:, end:end + HORIZON]
    scopes = {'ALL': np.ones(len(sales), bool)} | {c: (sales['cat_id'] == c).to_numpy() for c in ['FOODS', 'HOUSEHOLD', 'HOBBIES']}
    scorers = {s: WRMSSE(train_wide[m], calendar, prices) for s, m in scopes.items()}

    methods = {name: (lambda f=f, end=end: (f(end),)) for name, f in BASELINES.items()}
    methods |= {name: (lambda run=run, end=end: run(end)) for name, run in MODELS.items()}
    for name, run in methods.items():
        start = time.time()
        pred = run()[0]
        row = {'fold': f'd_{end + 1}–d_{end + HORIZON}', 'method': name, 'train_min': (time.time() - start) / 60,
               'bias_%': (pred.sum() / y_true.sum() - 1) * 100}
        row |= {s: scorers[s].score(y_true[m], pred[m]) for s, m in scopes.items()}
        rows.append(row)
        print(f"{row['fold']}  {name:15s} WRMSSE {row['ALL']:.3f}  ({row['train_min']:.1f} min)")
        if end == FOLDS[-1]:
            last_fold[name] = pred.astype(np.float32)

backtest = pd.DataFrame(rows)

---
## 2.6 ผลลัพธ์

### 2.6.1 ภาพใหญ่: วิธีไหนแม่นที่สุด

ค่าเฉลี่ย WRMSSE ของ 3 รอบ เส้นประคือค่า 1.0 (เท่ากับวิธี naive) ยิ่งแท่งสั้นยิ่งดี

In [ ]:
comparison = (
    backtest.groupby('method')[['ALL', 'FOODS', 'HOUSEHOLD', 'HOBBIES', 'bias_%', 'train_min']].mean()
    .rename(columns={'ALL': 'WRMSSE'})
    .sort_values('WRMSSE')
)
comparison.to_csv(ANALYSIS / 'model_comparison.csv')

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
colors = ['seagreen' if m in MODELS else 'gray' for m in comparison.index]
comparison['WRMSSE'].plot.barh(ax=axes[0], color=colors)
axes[0].axvline(1, color='black', ls='--', lw=1)
axes[0].invert_yaxis()
axes[0].bar_label(axes[0].containers[0], fmt='%.3f')
axes[0].set(title='WRMSSE, average of 3 backtests (lower is better)', xlabel='WRMSSE', ylabel='')

per_fold = backtest.pivot(index='fold', columns='method', values='ALL')[comparison.index]
per_fold.T.plot.bar(ax=axes[1], rot=30)
axes[1].set(title='WRMSSE by backtest round (stable across time?)', xlabel='', ylabel='WRMSSE')
show(fig, '02_6_model_comparison')
comparison

### 2.6.2 เลือกโมเดล

เกณฑ์: เลือกตัวที่ WRMSSE ต่ำที่สุด ถ้ามีตัวอื่นแม่นใกล้เคียงกัน (ห่างไม่เกิน 1%) ให้เลือกตัวที่เทรนเร็วกว่า

In [ ]:
best_score = comparison['WRMSSE'].min()
best = comparison[comparison['WRMSSE'] <= best_score * 1.01].sort_values('train_min').index[0]
baseline = comparison.loc[list(BASELINES), 'WRMSSE'].idxmin()
improvement = (1 - comparison.loc[best, 'WRMSSE'] / comparison.loc[baseline, 'WRMSSE']) * 100
print(f'โมเดลที่เลือก: {best} (WRMSSE {comparison.loc[best, "WRMSSE"]:.3f})')
print(f'baseline ที่ดีที่สุด: {baseline} (WRMSSE {comparison.loc[baseline, "WRMSSE"]:.3f})')
print(f'แม่นขึ้นจาก baseline {improvement:.1f}%')

### 2.6.3 Drill down: แยกตามหมวดสินค้าและ bias

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
comparison[['FOODS', 'HOUSEHOLD', 'HOBBIES']].plot.bar(ax=axes[0], rot=30)
axes[0].set(title='WRMSSE by category (lower is better)', xlabel='', ylabel='WRMSSE')
comparison['bias_%'].plot.bar(ax=axes[1], rot=30, color=np.where(comparison['bias_%'] > 0, 'steelblue', 'indianred'))
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set(title='Bias: total forecast vs actual (%)  — negative = under-forecast', xlabel='', ylabel='%')
show(fig, '02_6_by_category_bias')

### 2.6.4 Drill down: แยก 12 ระดับของ hierarchy (รอบ backtest ล่าสุด)

ระดับ 1 คือยอดรวมทั้งบริษัท ระดับ 12 คือรายสินค้า×สาขา ปกติระดับล่างจะยากกว่า เพราะยอดขายรายตัวผันผวนมาก

In [ ]:
LEVEL_NAMES = ['Total', 'State', 'Store', 'Category', 'Dept', 'State×Cat', 'State×Dept', 'Store×Cat',
               'Store×Dept', 'Item', 'Item×State', 'Item×Store']
end = FOLDS[-1]
y_true = Y[:, end:end + HORIZON]
scorer = WRMSSE(sales[ID_COLS + d_cols[:end]], calendar, prices)
by_level = pd.DataFrame({name: scorer.score(y_true, last_fold[name], by_level=True)[1] for name in [baseline, best]})
by_level.index = [f'{i}. {n}' for i, n in zip(by_level.index, LEVEL_NAMES)]
by_level.to_csv(ANALYSIS / 'wrmsse_by_level.csv')

fig, ax = plt.subplots(figsize=(12, 4.5))
by_level.plot.bar(ax=ax, rot=45, color=['gray', 'seagreen'])
ax.set(title=f'WRMSSE by hierarchy level, backtest d_{end + 1}–d_{end + HORIZON}', ylabel='WRMSSE')
show(fig, '02_6_by_level')

### 2.6.5 Drill down: ทายได้ใกล้ของจริงแค่ไหน (รอบ backtest ล่าสุด)

เปรียบเทียบยอดจริง (สีดำ) กับค่าที่ทาย จากภาพรวมทั้งบริษัท → รายหมวด → สินค้ารายตัว

In [ ]:
test_dates = dates[end:end + HORIZON]
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
ax_total = plt.subplot2grid((2, 3), (0, 0), colspan=3, fig=fig)
for ax in axes[0]:
    ax.remove()
ax_total.plot(test_dates, y_true.sum(axis=0), color='black', lw=2, label='Actual')
ax_total.plot(test_dates, last_fold[best].sum(axis=0), color='seagreen', lw=2, label=best)
ax_total.plot(test_dates, last_fold[baseline].sum(axis=0), color='gray', ls='--', label=baseline)
ax_total.set(title='Total company: daily units sold', ylabel='Units')
ax_total.legend()
for ax, cat in zip(axes[1], ['FOODS', 'HOUSEHOLD', 'HOBBIES']):
    m = (sales['cat_id'] == cat).to_numpy()
    ax.plot(test_dates, y_true[m].sum(axis=0), color='black', lw=2)
    ax.plot(test_dates, last_fold[best][m].sum(axis=0), color='seagreen', lw=2)
    ax.plot(test_dates, last_fold[baseline][m].sum(axis=0), color='gray', ls='--')
    ax.set(title=cat)
for ax in [ax_total, *axes[1]]:
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.SU))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
show(fig, '02_6_actual_vs_forecast_total')

In [ ]:
history_zero = (Y[:, end - 365:end] == 0).mean(axis=1)
revenue_rank = (Y[:, end - 28:end] * np.nan_to_num(fb.price[:, end - 28:end])).sum(axis=1)
picks = {
    'Fast-moving': int(np.argmax(np.where(history_zero < 0.3, revenue_rank, -1))),
    'Medium': int(np.argmax(np.where((history_zero >= 0.3) & (history_zero < 0.7), revenue_rank, -1))),
    'Slow-moving': int(np.argmax(np.where(history_zero >= 0.7, revenue_rank, -1))),
}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (label, i) in zip(axes, picks.items()):
    ax.bar(test_dates, y_true[i], color='lightgray', label='Actual')
    ax.plot(test_dates, last_fold[best][i], color='seagreen', lw=2, label=best)
    ax.plot(test_dates, last_fold[baseline][i], color='gray', ls='--', label=baseline)
    ax.set(title=f"{label}: {sales.loc[i, 'item_id']} @ {sales.loc[i, 'store_id']}", ylabel='Units')
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.SU))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b'))
axes[0].legend(fontsize=8)
show(fig, '02_6_actual_vs_forecast_items')

np.savez_compressed(
    PROCESSED / 'backtest_last_fold.npz',
    end=end, best=best, baseline=baseline, y_true=y_true,
    pred_best=last_fold[best], pred_baseline=last_fold[baseline],
    price=fb.price[:, end:end + HORIZON], cat_id=sales['cat_id'].to_numpy(),
    item_id=sales['item_id'].to_numpy(), store_id=sales['store_id'].to_numpy(),
)

**ข้อสังเกต:** ระดับรวมทายได้ใกล้เคียงมาก เพราะความผันผวนรายสินค้าหักล้างกันเมื่อรวมกัน
ระดับสินค้ารายตัว โมเดลทายเป็น "ค่าคาดหวัง" ที่เรียบกว่ายอดจริง โดยเฉพาะสินค้าขายช้าที่ยอดจริงเป็น 0 หรือ 1 สลับกัน ซึ่งเป็นธรรมชาติของสินค้ากลุ่มนี้

---
## 2.7 พยากรณ์จริง 28 วันข้างหน้า และอธิบายโมเดล

เทรนโมเดลที่เลือกใหม่ด้วยข้อมูลทั้งหมดถึง d_1941 (22 พ.ค. 2016) แล้วพยากรณ์ d_1942–d_1969 (23 พ.ค. – 19 มิ.ย. 2016)

In [ ]:
if best in MODELS:
    future, model, X_future = MODELS[best](FINAL_END)
else:
    future, model, X_future = BASELINES[best](FINAL_END), None, None

future_dates = dates[FINAL_END:FINAL_END + HORIZON]
forecast = pd.DataFrame(future, columns=pd.DatetimeIndex(future_dates).strftime('%Y-%m-%d'))
forecast.insert(0, 'store_id', sales['store_id'])
forecast.insert(0, 'item_id', sales['item_id'])
forecast.to_parquet(PROCESSED / 'forecast_next_28_days.parquet', index=False)

hist = slice(FINAL_END - 120, FINAL_END)
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
axes[0].plot(dates[hist], Y[:, hist].sum(axis=0), color='black', label='Actual (history)')
axes[0].plot(future_dates, future.sum(axis=0), color='seagreen', lw=2, label=f'Forecast ({best})')
axes[0].axvline(dates[FINAL_END], color='gray', ls=':')
axes[0].set(title='Total company: last 120 days + next 28 days forecast', ylabel='Units')
axes[0].legend()
for cat, color in zip(['FOODS', 'HOUSEHOLD', 'HOBBIES'], ['C0', 'C1', 'C2']):
    m = (sales['cat_id'] == cat).to_numpy()
    axes[1].plot(dates[hist], Y[m][:, hist].sum(axis=0), color=color, lw=1, label=cat)
    axes[1].plot(future_dates, future[m].sum(axis=0), color=color, lw=2.5, ls='--')
axes[1].axvline(dates[FINAL_END], color='gray', ls=':')
axes[1].set(title='By category (dashed = forecast)', ylabel='Units')
axes[1].legend()
show(fig, '02_7_future_forecast')

### อธิบายโมเดลด้วย SHAP

SHAP บอกว่าแต่ละ feature ดันค่าพยากรณ์ขึ้นหรือลงเท่าไร
- **แกนตั้ง:** feature เรียงจากสำคัญมากไปน้อย
- **จุดแต่ละจุด:** 1 แถวข้อมูล สีแดงคือค่า feature สูง สีน้ำเงินคือค่าต่ำ
- **ถ้าจุดอยู่ทางขวา:** feature นั้นดันยอดพยากรณ์ให้สูงขึ้น

In [ ]:
if model is not None:
    X_sample = X_future.sample(5_000, random_state=42)
    shap_values = shap.TreeExplainer(model).shap_values(X_sample)
    shap.summary_plot(shap_values, X_sample, max_display=15, show=False, plot_size=(10, 6))
    plt.title(f'{best}: what drives the forecast (SHAP)')
    show(plt.gcf(), '02_7_shap')

---
## 2.8 สรุปขั้นที่ 2

SUMMARY_PLACEHOLDER